# Instrument activity threshold inspector

Choose one or more instruments and a dBFS threshold. The notebook reads every
matching 96 kHz master stem, computes a 0.5-second RMS curve, and creates a
separate plot for every instrument/song pair.

Every run is saved in a new timestamped Drive folder. It contains PNG plots,
the raw time/RMS/threshold/activity CSV for every stem, per-instrument summary
CSVs, one combined summary CSV, and run metadata. Existing runs are never
overwritten.

This notebook is diagnostic only: it does not train a model or modify labels.

This repository copy recognizes audio files with or without a `.wav` suffix and
keeps take 2.2 distinct from song 2. It is based on the [original Colab notebook](https://colab.research.google.com/drive/1qstlrh5Q3jQFnDFnOvUVrJU1rwrb9iHx).


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import re
import unicodedata

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display, clear_output


# The 96 kHz masters are the common label source for all bandwidth experiments.
DRIVE_96_ROOT = Path("/content/drive/MyDrive/train_96")
RMS_WINDOW_SECONDS = 0.5
RMS_HOP_SECONDS = 0.5
DEFAULT_THRESHOLD_DBFS = -70.0
AUDIO_SUFFIXES = {".wav", ".flac", ".aif", ".aiff"}
OUTPUT_ROOT = Path("/content/drive/MyDrive/96k project/threshold_inspections")
LOCAL_TIMEZONE = ZoneInfo("Asia/Jerusalem")


def normalize_name(value: str) -> str:
    value = unicodedata.normalize("NFKC", value).casefold()
    return "".join(ch for ch in value if ch.isalnum())


def safe_filename(value: str) -> str:
    value = unicodedata.normalize("NFKC", str(value)).strip()
    value = re.sub(r"[^\w.-]+", "_", value, flags=re.UNICODE)
    return value.strip("._") or "item"


def create_run_directory(root: Path) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(LOCAL_TIMEZONE).strftime("%Y-%m-%d_%H-%M-%S")
    candidate = root / timestamp
    suffix = 1
    while candidate.exists():
        candidate = root / f"{timestamp}_{suffix:02d}"
        suffix += 1
    candidate.mkdir(parents=True)
    return candidate


def strip_audio_extension(filename: str) -> str:
    suffix = Path(filename).suffix
    return filename[:-len(suffix)] if suffix.casefold() in AUDIO_SUFFIXES else filename


def extract_song_id(filename: str):
    # Keep song 2.2 distinct from song 2; many Drive masters have no .wav suffix.
    match = re.search(r"(\d+(?:\.\d+)?)$", strip_audio_extension(filename))
    return match.group(1) if match else None


def song_sort_key(song_id: str):
    return tuple(int(component) for component in song_id.split("."))


def is_audio_file(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        sf.info(str(path))
        return True
    except Exception:
        return False


def instrument_directories(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"96 kHz root not found: {root}")
    result = {}
    for directory in sorted(path for path in root.iterdir() if path.is_dir()):
        if any(is_audio_file(path) for path in directory.iterdir()):
            result[directory.name] = directory
    if not result:
        raise FileNotFoundError(
            f"No instrument folders with readable audio were found under {root}"
        )
    return result


def stem_files_for_instrument(root: Path, instrument: str):
    directories = instrument_directories(root)
    matches = [
        directory
        for name, directory in directories.items()
        if normalize_name(name) == normalize_name(instrument)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one folder for {instrument!r}, found {[path.name for path in matches]}"
        )
    files = []
    for path in sorted(matches[0].iterdir()):
        if not is_audio_file(path):
            continue
        song_id = extract_song_id(path.name)
        if song_id is not None:
            files.append((song_id, path))
    return sorted(files, key=lambda item: (song_sort_key(item[0]), item[1].name.casefold()))


def streaming_rms_dbfs(path: Path, window_seconds: float, hop_seconds: float):
    """Compute RMS without loading a whole 96 kHz song into memory."""
    with sf.SoundFile(str(path), mode="r") as handle:
        sample_rate = int(handle.samplerate)
        total_frames = int(len(handle))
        window_frames = max(1, int(round(window_seconds * sample_rate)))
        hop_frames = max(1, int(round(hop_seconds * sample_rate)))
        times = []
        values = []
        for start in range(0, total_frames, hop_frames):
            handle.seek(start)
            block = handle.read(window_frames, dtype="float32", always_2d=True)
            if len(block) == 0:
                break
            mono = np.mean(block, axis=1, dtype=np.float32)
            rms = float(np.sqrt(np.mean(np.square(mono, dtype=np.float64))))
            values.append(20.0 * np.log10(max(rms, 1e-10)))
            times.append((start + 0.5 * len(block)) / sample_rate)
        return np.asarray(times), np.asarray(values), sample_rate


def inspect_instrument(instrument: str, threshold_dbfs: float, run_directory: Path):
    files = stem_files_for_instrument(DRIVE_96_ROOT, instrument)
    if not files:
        raise FileNotFoundError(f"No song stems found for {instrument!r}")

    instrument_directory = run_directory / safe_filename(instrument)
    plot_directory = instrument_directory / "plots"
    rms_directory = instrument_directory / "rms_csv"
    plot_directory.mkdir(parents=True, exist_ok=True)
    rms_directory.mkdir(parents=True, exist_ok=True)

    rows = []
    for song_id, path in files:
        times, rms_dbfs, sample_rate = streaming_rms_dbfs(
            path, RMS_WINDOW_SECONDS, RMS_HOP_SECONDS
        )
        active = rms_dbfs >= threshold_dbfs
        activity_percent = float(100.0 * np.mean(active)) if len(active) else 0.0
        stem_name = safe_filename(strip_audio_extension(path.name))
        plot_path = plot_directory / f"song_{song_id.replace('.', '_')}_{stem_name}.png"
        rms_path = rms_directory / f"song_{song_id.replace('.', '_')}_{stem_name}.csv"

        pd.DataFrame(
            {
                "time_seconds": times,
                "rms_dbfs": rms_dbfs,
                "threshold_dbfs": threshold_dbfs,
                "active": active.astype(np.uint8),
            }
        ).to_csv(rms_path, index=False)

        rows.append(
            {
                "instrument": instrument,
                "song_id": song_id,
                "file": path.name,
                "sample_rate": sample_rate,
                "windows": len(rms_dbfs),
                "threshold_dbfs": threshold_dbfs,
                "activity_percent": activity_percent,
                "rms_csv": str(rms_path),
                "plot_png": str(plot_path),
            }
        )

        fig, ax = plt.subplots(figsize=(16, 4.5))
        ax.plot(
            times,
            rms_dbfs,
            linewidth=1.0,
            color="#1f77b4",
            label="0.5 s RMS",
        )
        ax.axhline(
            threshold_dbfs,
            color="#d62728",
            linestyle="--",
            linewidth=1.5,
            label=f"Threshold {threshold_dbfs:.1f} dBFS",
        )
        ax.fill_between(
            times,
            rms_dbfs,
            threshold_dbfs,
            where=active,
            color="#2ca02c",
            alpha=0.18,
            interpolate=True,
            label="Above threshold",
        )
        ax.set_title(
            f"{instrument} — song {song_id} — active {activity_percent:.1f}%"
        )
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("RMS (dBFS)")
        ax.set_ylim(max(-120.0, float(np.min(rms_dbfs)) - 3.0), 0.0)
        ax.grid(True, alpha=0.25)
        ax.legend(loc="best")
        fig.tight_layout()
        fig.savefig(plot_path, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    summary = pd.DataFrame(rows).reset_index(drop=True)
    summary.to_csv(instrument_directory / "activity_summary.csv", index=False)
    display(summary)
    return summary


In [ ]:
directories = instrument_directories(DRIVE_96_ROOT)
instrument_options = sorted(directories)
default_instrument = "Drums" if "Drums" in instrument_options else instrument_options[0]

instrument_widget = widgets.SelectMultiple(
    options=instrument_options,
    value=(default_instrument,),
    description="Instruments:",
    style={"description_width": "initial"},
    rows=min(12, max(5, len(instrument_options))),
    layout=widgets.Layout(width="600px"),
)
threshold_widget = widgets.FloatSlider(
    value=DEFAULT_THRESHOLD_DBFS,
    min=-120.0,
    max=-20.0,
    step=1.0,
    description="Threshold dBFS:",
    style={"description_width": "initial"},
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="600px"),
)
selection_help = widgets.HTML(
    "Select one or more instruments (Ctrl/Command-click), then click Run and save."
)
run_button = widgets.Button(description="Run and save", button_style="primary")
output = widgets.Output()


def run_inspection(_):
    instruments = list(instrument_widget.value)
    if not instruments:
        with output:
            clear_output(wait=True)
            print("Select at least one instrument.")
        return

    run_button.disabled = True
    try:
        with output:
            clear_output(wait=True)
            threshold_dbfs = float(threshold_widget.value)
            run_directory = create_run_directory(OUTPUT_ROOT)
            print(f"Saving this run to: {run_directory}")
            print(f"Selected instruments: {', '.join(instruments)}")
            print(f"Threshold: {threshold_dbfs:.1f} dBFS")

            summaries = []
            for index, instrument in enumerate(instruments, start=1):
                print(f"\n[{index}/{len(instruments)}] {instrument}")
                summaries.append(
                    inspect_instrument(instrument, threshold_dbfs, run_directory)
                )

            combined_summary = pd.concat(summaries, ignore_index=True)
            combined_path = run_directory / "activity_summary.csv"
            combined_summary.to_csv(combined_path, index=False)

            metadata = {
                "created_at": datetime.now(LOCAL_TIMEZONE).isoformat(),
                "instruments": instruments,
                "threshold_dbfs": threshold_dbfs,
                "rms_window_seconds": RMS_WINDOW_SECONDS,
                "rms_hop_seconds": RMS_HOP_SECONDS,
                "source_root": str(DRIVE_96_ROOT),
                "output_directory": str(run_directory),
            }
            (run_directory / "run_metadata.json").write_text(
                json.dumps(metadata, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            print(f"\nDone. Combined summary: {combined_path}")
            display(combined_summary)
    finally:
        run_button.disabled = False


run_button.on_click(run_inspection)
display(
    widgets.VBox(
        [selection_help, instrument_widget, threshold_widget, run_button]
    ),
    output,
)
